# Hito 2: Creación, Entrenamiento y Validación de Modelos (Machine Learning)
## Proyecto: HedgeMind-NVDA
**Autor:** Fausto Jara Buncay  
**Problema:** Regresión (Predicción de rentabilidad futura `Target_Return`)

Este cuaderno aborda la Etapa 5 (Creación y ajuste de hiperparámetros) y la Etapa 6 (Validación con métricas MAE, RMSE y R²). Para respetar la naturaleza cronológica de los datos bursátiles, se aplicará `TimeSeriesSplit` en lugar de validación cruzada estándar.

In [20]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor # Asegúrate de instalarlo con: pip install xgboost
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Recuperar los datos limpios directamente de AWS
load_dotenv()
conexion_str = f"mysql+mysqlconnector://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
engine = create_engine(conexion_str)

print("☁️ Descargando dataset consolidado desde Amazon RDS...")
df = pd.read_sql("SELECT * FROM dataset_nvda_unificado", con=engine, index_col='fecha')
df.index = pd.to_datetime(df.index)

# 2. Recrear rápidamente las variables (Feature Engineering)
df['Log_Volume'] = np.log1p(df['trading_volume'])
df['Price_Square'] = df['close_price'] ** 2
df['Sentiment_Impact'] = df['sentiment_score'] * df['news_volume']
df.dropna(inplace=True)

# 3. Separación temporal estricta (80% Train, 20% Test)
X = df.drop(columns=['target_return'])
y = df['target_return']
split_idx = int(len(df) * 0.8)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# 4. Escalado (Ajustado solo en Train)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Datos listos. Registros de Entrenamiento: {len(X_train)} | Test: {len(X_test)}")

☁️ Descargando dataset consolidado desde Amazon RDS...
✅ Datos listos. Registros de Entrenamiento: 5489 | Test: 1373


## 5. Creación y Entrenamiento de Modelos
Se evaluarán dos arquitecturas de ensamble: **Random Forest** (basado en Bagging) y **XGBoost** (basado en Boosting). Se utilizará `GridSearchCV` junto con `TimeSeriesSplit` para optimizar los hiperparámetros sin incurrir en *Data Leakage*.

In [21]:
# ==============================================================================
# MODELO 1: RANDOM FOREST REGRESSOR
# ==============================================================================
print("🌲 Iniciando entrenamiento de Random Forest...")

# Definimos el validador cruzado para series temporales (5 divisiones cronológicas)
tscv = TimeSeriesSplit(n_splits=5)

# Modelo base
rf_base = RandomForestRegressor(random_state=42, n_jobs=-1) # n_jobs=-1 usa todos los núcleos de tu PC

# Malla de hiperparámetros a explorar
param_grid_rf = {
    'n_estimators': [50, 100],        # Cantidad de árboles
    'max_depth': [5, 10, None],       # Profundidad máxima (evita overfitting)
    'min_samples_split': [2, 5]       # Mínimo de muestras para dividir un nodo
}

# Búsqueda en cuadrícula (GridSearch)
grid_rf = GridSearchCV(estimator=rf_base, param_grid=param_grid_rf, 
            cv=tscv, scoring='neg_mean_squared_error', n_jobs=-1)

grid_rf.fit(X_train_scaled, y_train)

best_rf = grid_rf.best_estimator_
print(f"✅ Random Forest entrenado. Mejores hiperparámetros: {grid_rf.best_params_}")

🌲 Iniciando entrenamiento de Random Forest...
✅ Random Forest entrenado. Mejores hiperparámetros: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 100}


In [22]:
# ==============================================================================
# MODELO 2: EXTREME GRADIENT BOOSTING (XGBoost)
# ==============================================================================
print("🚀 Iniciando entrenamiento de XGBoost...")

# Modelo base (Activamos la aceleración por GPU ya que tienes una RTX 4060)
# Si te da error por la GPU, cambia 'cuda' por 'cpu'
xgb_base = XGBRegressor(random_state=42, device='cuda') 

# Malla de hiperparámetros
param_grid_xgb = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1],     # Tasa de aprendizaje
    'max_depth': [3, 6],              # Árboles más pequeños que en RF
    'subsample': [0.8, 1.0]           # Fracción de datos usada por árbol
}

grid_xgb = GridSearchCV(estimator=xgb_base, param_grid=param_grid_xgb, 
                        cv=tscv, scoring='neg_mean_squared_error')

grid_xgb.fit(X_train_scaled, y_train)

best_xgb = grid_xgb.best_estimator_
print(f"✅ XGBoost entrenado. Mejores hiperparámetros: {grid_xgb.best_params_}")

🚀 Iniciando entrenamiento de XGBoost...
✅ XGBoost entrenado. Mejores hiperparámetros: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}


## 6. Validación de los Modelos
A continuación, calculamos las métricas solicitadas (MAE, RMSE y R²) utilizando el conjunto de Test (datos que los modelos nunca han visto) para determinar cuál posee una mayor bondad de ajuste.

In [23]:
# ==============================================================================
# EVALUACIÓN Y SELECCIÓN DEL MEJOR MODELO
# ==============================================================================
# Predicciones con el conjunto de validación
y_pred_rf = best_rf.predict(X_test_scaled)
y_pred_xgb = best_xgb.predict(X_test_scaled)

def calcular_metricas(y_real, y_pred, nombre_modelo):
    mae = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    r2 = r2_score(y_real, y_pred)
    return pd.Series({'MAE': mae, 'RMSE': rmse, 'R2': r2}, name=nombre_modelo)

# Consolidar métricas en una tabla comparativa
metricas_rf = calcular_metricas(y_test, y_pred_rf, 'Random Forest')
metricas_xgb = calcular_metricas(y_test, y_pred_xgb, 'XGBoost')

df_metricas = pd.concat([metricas_rf, metricas_xgb], axis=1)

print("=== COMPARATIVA DE BONDAD DE AJUSTE EN EL CONJUNTO DE TEST ===")
print(df_metricas.round(6))

print("\nInterpretación Académica:")
print("- MAE (Error Absoluto Medio): Cuánto nos equivocamos en promedio en la predicción del porcentaje de retorno.")
print("- RMSE (Raíz del Error Cuadrático Medio): Penaliza severamente los errores de predicción muy grandes.")
print("- R² (Coeficiente de Determinación): Qué porcentaje de la varianza del mercado logramos explicar. (En finanzas, valores > 0.05 ya se consideran valiosos debido a la alta aleatoriedad del mercado).")

=== COMPARATIVA DE BONDAD DE AJUSTE EN EL CONJUNTO DE TEST ===
      Random Forest   XGBoost
MAE        0.037744  0.023559
RMSE       0.046414  0.032113
R2        -1.094684 -0.002730

Interpretación Académica:
- MAE (Error Absoluto Medio): Cuánto nos equivocamos en promedio en la predicción del porcentaje de retorno.
- RMSE (Raíz del Error Cuadrático Medio): Penaliza severamente los errores de predicción muy grandes.
- R² (Coeficiente de Determinación): Qué porcentaje de la varianza del mercado logramos explicar. (En finanzas, valores > 0.05 ya se consideran valiosos debido a la alta aleatoriedad del mercado).


In [24]:
import pandas as pd
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv()
engine = create_engine(f"mysql+mysqlconnector://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:3306/{os.getenv('DB_NAME')}")

# Pedimos a la base de datos que nos enseñe 5 días donde SÍ hubo noticias reales
query = "SELECT fecha, close_price, sentiment_score, news_volume FROM dataset_nvda_unificado WHERE news_volume > 0 LIMIT 5"
print(pd.read_sql(query, con=engine))

       fecha  close_price  sentiment_score  news_volume
0 2020-06-02     8.790784         0.220200          2.0
1 2020-06-08     8.774614         0.168667          3.0
2 2020-06-09     9.014781         0.729600          1.0
3 2020-06-10     9.334425         0.243200          3.0
